# SANOS-KF — fixed model: run & evaluate

Runs the repaired `SANOS_KF` model (forward re-centering · correlated RBF process noise ·
per-second batching, with the `smoothing` flag) and scores it with the unified metric set:

* **in-sample** (post-update fit) and **held-out** (one-step-ahead prior) — both for
* **NSE** = mean over a snapshot's strikes of |model − mid| / half-spread, and
* **RMSE** in undiscounted call-price dollars,

each reported as **mean / median / std** over the 1-second snapshots.

## Imports

In [ ]:
import os
import sys
from datetime import date

import numpy as np
import pandas as pd
import polars as pl

project_root = "/Users/elie/Documents/GitHub/quant_data/"
for p in [
    project_root,
    os.path.join(project_root, "personal", "Kalman_filter_spline"),
    os.path.join(project_root, "personal", "shared_functions"),
]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ.setdefault("QUANTDATA_RAW_DATA", "/quantdata/raw")

import personal.shared_functions.unified_metrics as um
from personal.Kalman_filter_spline.surface_kalman_filters import build_iv_observations
from personal.SANOS_KF.sanos_kf import (
    SanosKFConfig,
    build_exact_grid,
    exact_solve_density,
    run_sanos_kf,
)
from quantdata.massive.client import MassiveClient

## Config & data

In [2]:
TICKER = "SPY"
TRADE_DATE = date(2024, 1, 3)
MIN_TRADES = 100
BATCH_S = 1.0  # 1-second snapshots (must match cfg.batch_seconds below)

df_tickers = pl.read_csv(os.path.join(project_root, "personal", "spy_options_tickers.csv"))
client = MassiveClient()
df_spy = client.fetch_trades(TICKER, TRADE_DATE)

df_quotes_all, df_trades_all = build_iv_observations(
    df_tickers, TRADE_DATE, df_spy, min_trades=MIN_TRADES
)
print(f"quotes: {len(df_quotes_all):,}   trades: {len(df_trades_all):,}")
print("columns:", df_quotes_all.columns)

Skipping O:SPY240105C00481000: 46 bad quotes
Skipping O:SPY240105C00482000: 64 bad quotes
Skipping O:SPY240105C00483000: 842 bad quotes
Skipping O:SPY240105C00484000: 2129 bad quotes
Skipping O:SPY240105C00485000: 3888 bad quotes
Skipping O:SPY240105P00462500: 17 bad quotes
Expiry 2024-01-05: implied forward from 13 strike(s)
quotes: 2,115,510   trades: 74,099
columns: ['ts_ns', 'strike', 'is_call', 'log_moneyness', 'T', 'observed_iv', 'bid_iv', 'ask_iv', 'R_iv', 'bid_price', 'ask_price', 'forward', 'discount_factor']


## Eval helper

`run_sanos_kf` outputs **normalized** call prices (`sanos_kf_price` = posterior,
`sanos_kf_price_prior` = held-out prior, both = price/(F·df)). We:

1. tag each quote with its 1-second `window` and keep the **last** quote per
   `(window, strike, is_call)` as that snapshot's representative (state is constant
   within a window, so this avoids over-weighting strikes that printed more often);
2. map normalized call → undiscounted native option price (`C/df = h·F`, puts via
   reverse parity) and feed `um.from_option_prices` / `um.evaluate`.

**Arbitrage:** in exact-solve mode the negative mass is computed on the *model's own
density* (`Σ min(d,0)` per snapshot, via `exact_solve_density`), not via
Breeden–Litzenberger on the sparse market strikes — the latter conflates genuine
negativity with extrapolation outside the anchor grid and finite-difference noise.

In [ ]:
def run_and_eval(cfg, model_name):
    """Run the fixed SANOS-KF and return um.evaluate's metric dict."""
    q, _, st = run_sanos_kf(df_quotes_all, df_trades_all, cfg, return_state=True)
    qb = q.to_pandas()

    bs_ns = int((cfg.batch_seconds or BATCH_S) * 1e9)
    qb["window"] = qb["ts_ns"].astype("int64") // bs_ns
    qb = (
        qb.sort_values("ts_ns")
        .drop_duplicates(["window", "strike", "is_call"], keep="last")
        .dropna(subset=["sanos_kf_price_prior"])
    )

    F = qb["forward"].to_numpy(float)
    K = qb["strike"].to_numpy(float)
    ic = qb["is_call"].to_numpy(bool)
    dfac = qb["discount_factor"].to_numpy(float) if "discount_factor" in qb else np.ones(len(qb))

    # sanos_kf_price is F-normalized call (= price/(F·df)) in EVERY mode (the exact-solve
    # path divides its absolute call by F before output). ×F -> undiscounted call (C/df),
    # then to_native -> native option price. Same convention as the other KF models.
    to_native = lambda c: np.where(ic, c, c - (F - K))  # noqa: E731
    prior_native = to_native(qb["sanos_kf_price_prior"].to_numpy(float) * F)
    post_native = to_native(qb["sanos_kf_price"].to_numpy(float) * F)

    df_eval = um.from_option_prices(
        ts=qb["window"].to_numpy(),
        strike=K,
        forward=F,
        is_call=ic,
        bid=qb["bid_price"].to_numpy(float),
        ask=qb["ask_price"].to_numpy(float),
        prior_price=prior_native,
        post_price=post_native,
        discount_factor=dfac,
    )

    # Arbitrage: exact-solve scores negative mass on the MODEL's own density
    # (Σ min(d,0) per snapshot, on the absolute strike grid) — not a Breeden-
    # Litzenberger re-derivation from the sparse market strikes.
    neg_vals = None
    exact = (not cfg.smoothing) and (float(cfg.sigma_star_init) == 0.0)
    if exact:
        grid_K, tte_ref = build_exact_grid(df_quotes_all, cfg)
        F0 = float(np.nanmedian(df_quotes_all["forward"].to_numpy().astype(float)))
        stp = st.to_pandas()
        stp["window"] = stp["ts_ns"].astype("int64") // bs_ns
        stp = stp.sort_values("ts_ns").drop_duplicates("window", keep="last")
        xcols = [c for c in stp.columns if c.startswith("x_")]
        negs = []
        for xrow in stp[xcols].to_numpy(float):
            d = exact_solve_density(xrow, grid_K, tte_ref, cfg.dlv_min, fwd=F0)
            negs.append(np.nan if d is None else float(np.minimum(d, 0.0).sum()))
        neg_vals = np.array(negs)

    return um.evaluate(df_eval, model_name=model_name, neg_mass_values=neg_vals)

## Run the model

In [ ]:
# Market side: "both" | "calls" | "puts"
SIDE = "puts"
side_kw = {"calls": dict(calls_only=True), "puts": dict(puts_only=True), "both": {}}[SIDE]

# Exact-solve runs in ABSOLUTE STRIKE space (mimics SANOS_KF_Strike): strike-aligned
# grid, density re-centers via Σd·K=F. ell is in DOLLARS, q/Sigma/P0 on the one-phase
# scale. Defaults below = sweep best on SPY 2024-01-03 puts (ho_nse_med ≈ 0.48).
cfg = SanosKFConfig(
    n_anchors=60,  # caps strike-aligned interior nodes
    k_min=-0.60,
    k_max=0.10,  # cover the data's log-moneyness (≈ -0.56..+0.07)
    Sigma_init=0.01,  # flat net-DLV² init
    sigma_star_init=0.0,  # σ*=0 ⇒ exact-solve
    Q_scale_dlv=0.1,
    P0_scale=1.0,
    correlated=True,
    ell=3.0,  # RBF correlation length in DOLLARS (strike distance)
    smoothing=False,  # exact-solve density (no BS mixture)
    batch_seconds=BATCH_S,
    **side_kw,
)
m = run_and_eval(cfg, f"SANOS-KF exact-solve ({SIDE})")
m

## Requested metrics — in-sample / held-out · NSE & RMSE · mean / median / std

In [5]:
def metrics_frame(metric_dicts):
    rows = []
    for d in metric_dicts:
        for split, tag in [("ho", "held-out"), ("is", "in-sample")]:
            rows.append(
                {
                    "model": d["model"],
                    "split": tag,
                    "NSE mean": d[f"{split}_nse_mean"],
                    "NSE med": d[f"{split}_nse_med"],
                    "NSE std": d[f"{split}_nse_std"],
                    "RMSE mean": d[f"{split}_rmse_mean"],
                    "RMSE med": d[f"{split}_rmse_med"],
                    "RMSE std": d[f"{split}_rmse_std"],
                }
            )
    return (
        pd.DataFrame(rows)
        .set_index(["model", "split"])
        .round(
            {
                "NSE mean": 3,
                "NSE med": 3,
                "NSE std": 3,
                "RMSE mean": 4,
                "RMSE med": 4,
                "RMSE std": 4,
            }
        )
    )


metrics_frame([m])

NSE mean   NSE med  NSE std  RMSE mean  \
model                  split                                               
SANOS-KF (fixed, both) held-out   1614.280  1577.242  268.517    12.2652   
                       in-sample  1614.286  1577.242  268.494    12.2652   

                                  RMSE med  RMSE std  
model                  split                          
SANOS-KF (fixed, both) held-out    12.2653    0.2937  
                       in-sample   12.2654    0.2933

## Diagnostic — eval space & learning check

Confirms (a) prices are compared in **undiscounted dollars** (`price/df`), the same space
as the other models, and (b) whether the filter is actually *learning* — i.e. whether the
posterior moves away from the one-step-ahead prior. If `post ≈ prior` while `post ≠ mid`,
the problem is the model/conditioning, not the evaluation.

In [6]:
# One snapshot, undiscounted-dollar (price/df) space: mid vs one-step-ahead prior vs posterior.
q, _ = run_sanos_kf(df_quotes_all, df_trades_all, cfg)
qb = q.to_pandas()
qb["window"] = qb["ts_ns"].astype("int64") // int(BATCH_S * 1e9)
qb = (
    qb.sort_values("ts_ns")
    .drop_duplicates(["window", "strike", "is_call"], keep="last")
    .dropna(subset=["sanos_kf_price_prior"])
)

snap = qb[qb["window"] == qb["window"].max()].copy()
F = snap["forward"].to_numpy(float)
K = snap["strike"].to_numpy(float)
ic = snap["is_call"].to_numpy(bool)
dfac = snap["discount_factor"].to_numpy(float) if "discount_factor" in snap else np.ones(len(snap))
to_native = lambda c: np.where(ic, c, c - (F - K))  # normalized-call*F -> native undiscounted

diag = pd.DataFrame(
    {
        "K": K,
        "is_call": ic,
        "mid (px/df)": 0.5
        * (snap["bid_price"] + snap["ask_price"]).to_numpy(float)
        / dfac,  # native undisc
        "prior": to_native(snap["sanos_kf_price_prior"].to_numpy(float) * F),
        "post": to_native(snap["sanos_kf_price"].to_numpy(float) * F),
    }
).sort_values(["K", "is_call"])
diag["post-mid"] = diag["post"] - diag["mid (px/df)"]
diag["post-prior (learning)"] = diag["post"] - diag["prior"]
print("Deep-ITM call mid should equal intrinsic F-K (proves dollar/strike space).")
print(
    f"|post-prior| mean = {diag['post-prior (learning)'].abs().mean():.4f}  "
    f"(≈0 ⇒ filter barely updating)"
)
diag.round(3)

Deep-ITM call mid should equal intrinsic F-K (proves dollar/strike space).
|post-prior| mean = 0.0000  (≈0 ⇒ filter barely updating)


,K,is_call,mid (px/df),prior,post,post-mid,post-prior (learning)
0,466.0,True,3.805,16.382,16.382,12.577,0.0
1,469.0,True,1.885,14.731,14.731,12.846,0.0
2,472.0,False,3.690,16.153,16.153,12.463,0.0
3,474.0,False,5.330,17.244,17.244,11.914,0.0


## Variant comparison (smoothing on/off, exact-solve)

Same model, three settings:
* free σ\* (BS-mixture smoothing),
* frozen light smoothing (σ\*=0.05),
* **σ\*=0 ⇒ exact-solve density** (well-conditioned SANOS exactly-determined system,
  state = net DLV², the `SANOS_KF_Strike` analogue — *not* the BS-mixture intrinsic).

In [ ]:
# exact-solve here is ABSOLUTE strike space (ell in dollars). The BS-mixture modes
# (smoothing / frozen σ*) are normalized-moneyness; their ell/Sigma_init live on a
# different scale, so this comparison is mainly meaningful for the exact-solve row.
variants = [
    (dict(smoothing=False, sigma_star_init=0.0), "exact-solve (σ*=0, strike)"),
]
base = dict(
    n_anchors=60,
    k_min=-0.60,
    k_max=0.10,
    Sigma_init=0.01,
    Q_scale_dlv=0.1,
    P0_scale=1.0,
    correlated=True,
    ell=3.0,  # DOLLARS — sweep best on 2024-01-03 puts
    batch_seconds=BATCH_S,
    **side_kw,
)

dicts = [run_and_eval(SanosKFConfig(**{**base, **ov}), name) for ov, name in variants]
display(metrics_frame(dicts))
um.comparison_table(dicts)